# TalentCLEF TaskA 2025

# Carga de datos

In [2]:
import pandas as pd
import numpy as np

In [3]:
training_data = pd.read_csv("./data/TaskA/training/english/taskA_training_en.tsv", sep='\t', names=['family_id', 'id', 'title1', 'title2'])

validation_data = dict()

validation_data['corpus_elements'] = pd.read_csv("./data/TaskA/validation/english/corpus_elements", sep='\t').set_index('c_id').reset_index(drop=True)

validation_data['queries'] = pd.read_csv("./data/TaskA/validation/english/queries", sep='\t').set_index('q_id').reset_index(drop=True)

validation_data['qrels'] = pd.read_csv("./data/TaskA/validation/english/qrels.tsv", sep='\t', names=['q_id', 'iter', 'c_id', 'relevance'])[['q_id', 'c_id', 'relevance']]
validation_data['qrels']['q_id'] = validation_data['qrels']['q_id'] - 1
validation_data['qrels']['c_id'] = validation_data['qrels']['c_id'] - 1

In [4]:
validation_data['qrels']

,q_id,c_id,relevance
0,0,142,1
1,0,149,1
2,0,763,1
3,0,869,1
4,0,1463,1
...,...,...,...
2415,104,2122,1
2416,104,2143,1
2417,104,2355,1
2418,104,2399,1


In [5]:
print("Training data samples:")
print(training_data[['title1', 'title2']].head())
print("\n")

print("Validation data samples:")
print(validation_data['corpus_elements'].head())
print(validation_data['queries'].head())
print(validation_data['qrels'].head())

Training data samples:
                        title1                       title2
0                air commodore            flight lieutenant
1  command and control officer               flight officer
2                air commodore  command and control officer
3                pilot officer              squadron leader
4       royal airforce officer  command and control officer


Validation data samples:
                           jobtitle
0                recording engineer
1              director of taxation
2  technical support representative
3                        hr manager
4           computer graphic artist
              jobtitle
0                nanny
1    food technologist
2   broadcast engineer
3  automation engineer
4         veterinarian
   q_id  c_id  relevance
0     0   142          1
1     0   149          1
2     0   763          1
3     0   869          1
4     0  1463          1


# Aproximación con embedding ya preentrenado

Se establece device='cpu' debido a que mi ordenador no soporta CUDA

## Configuración

In [6]:
from sentence_transformers import SentenceTransformer, util

# Version multilingue del modelo
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', device='cpu')

/home/david/Documents/Master/TFM/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
def similarity_between_titles(title1, title2):
    emb1 = model.encode(title1, convert_to_tensor=True, device='cpu')
    emb2 = model.encode(title2, convert_to_tensor=True, device='cpu')

    similarity = util.cos_sim(emb1, emb2)

    return similarity.item()

similarity_between_titles("data scientist", "científico de datos")

0.963962197303772

In [8]:
print('Total de casos a revisar:', validation_data['corpus_elements'].shape[0] * validation_data['queries'].shape[0])

Total de casos a revisar: 274995


## Cálculo de similitud

In [22]:
# calcular la matriz de similitud de query x corpus elements
query_embeddings = model.encode(validation_data['queries']['jobtitle'].tolist()[:10], convert_to_tensor=True, device='cpu', show_progress_bar=True)
corpus_embeddings = model.encode(validation_data['corpus_elements']['jobtitle'].tolist()[:10], convert_to_tensor=True, device='cpu', show_progress_bar=True)

cosine_scores = util.cos_sim(query_embeddings, corpus_embeddings).cpu().numpy()
print('Matriz de similitud (query x corpus elements):', cosine_scores.shape)
print(cosine_scores)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  7.12it/s]

Matriz de similitud (query x corpus elements): (10, 10)
[[0.21296816 0.23104087 0.30833417 0.28416908 0.23594415 0.20110852
  0.14726932 0.10741566 0.16541019 0.2722712 ]
 [0.26919237 0.17486244 0.3027686  0.40574205 0.31700575 0.20039475
  0.27209136 0.12280918 0.20780316 0.34582296]
 [0.7295592  0.09044953 0.39284682 0.35019016 0.39105797 0.55269104
  0.20804837 0.13801937 0.17087182 0.13967463]
 [0.4464507  0.07589401 0.42346135 0.37309483 0.4513507  0.34924316
  0.26818117 0.11797051 0.11251827 0.27919066]
 [0.24932191 0.13233133 0.19779621 0.3490096  0.1692468  0.22204044
  0.19643036 0.11505149 0.2831126  0.14652759]
 [0.31029537 0.41361082 0.34389126 0.47828436 0.16761246 0.4149703
  0.5522942  0.22445387 0.22943485 0.4570803 ]
 [0.24514471 0.08376292 0.29009905 0.2779016  0.30750868 0.17789736
  0.19623533 0.02078242 0.22851115 0.15939653]
 [0.25412723 0.20409772 0.09464145 0.34299725 0.14918485 0.21989048
  0.277882   0.11440062 0.1555984  0.00889824]
 [0.1760211  0.37068456 0

# Generacion de datos sinteticos para testear la busqueda del threshold

In [15]:
import random

# Lista para almacenar las filas
data = []

# Para cada q_id de 0 a 9
for q_id in range(10):
    # Generar entre 2 y 4 c_id aleatorios
    num_elements = random.randint(1,3)
    c_ids = random.sample(range(10), num_elements)
    
    # Crear una fila por cada c_id
    for c_id in c_ids:
        data.append({
            'q_id': q_id,
            'c_id': c_id,
            'relevance': 1
        })

# Crear el DataFrame
temp_val = pd.DataFrame(data).sort_values(['q_id', 'c_id']).reset_index(drop=True)

print(temp_val)
print(f"\nTotal de filas: {len(temp_val)}")

    q_id  c_id  relevance
0      0     4          1
1      0     5          1
2      0     6          1
3      1     4          1
4      2     1          1
5      2     3          1
6      2     8          1
7      3     5          1
8      3     6          1
9      4     7          1
10     4     9          1
11     5     1          1
12     5     3          1
13     5     9          1
14     6     6          1
15     6     8          1
16     7     4          1
17     8     8          1
18     9     0          1
19     9     7          1

Total de filas: 20


In [ ]:
query_size = 10
corpus_size = 10

relevance_matrix = np.zeros((query_size, corpus_size), dtype=int)
for index, row in temp_val.iterrows():
    relevance_matrix[row['q_id'], row['c_id']] = row['relevance']

relevance_matrix

array([[0, 0, 0, 0, 1, 1, 1, 0, 0, 0],
       [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
       [0, 1, 0, 1, 0, 0, 0, 0, 1, 0],
       [0, 0, 0, 0, 0, 1, 1, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 1, 0, 1],
       [0, 1, 0, 1, 0, 0, 0, 0, 0, 1],
       [0, 0, 0, 0, 0, 0, 1, 0, 1, 0],
       [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
       [1, 0, 0, 0, 0, 0, 0, 1, 0, 0]])

In [24]:
y_true = relevance_matrix.flatten()
y_scores = cosine_scores.flatten()

In [26]:
from sklearn.metrics import accuracy_score

thresholds = np.linspace(0, 1, 1000)  # 200 valores entre 0 y 1
accuracies = []

for t in thresholds:
    preds = (y_scores >= t).astype(int)
    acc = accuracy_score(y_true, preds)
    accuracies.append(acc)

best_idx = np.argmax(accuracies)
best_threshold = thresholds[best_idx]
best_accuracy = accuracies[best_idx]

print(f"Threshold óptimo (accuracy): {best_threshold:.4f}")
print(f"Accuracy: {best_accuracy:.4f}")


Threshold óptimo (accuracy): 0.7377
Accuracy: 0.8000
